# Jeet Bengali voice adaptation
Private CLI run using `jeet065/jeet-voice`. A Kaggle-hosted copy of IndicF5 is verified against SHA-256 hashes of the pinned base weights and vocabulary before any training starts. Model-source Python files are never imported. GPU and Internet are enabled by the submission. No tokens are embedded.
45 training clips / 5 validation clips. Download best.safetensors and compare speech against the base model.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys, os, json, zipfile

DATASET_ID = 'jeet065/jeet-voice'
INPUT = Path('/kaggle/input')
roots = [p for p in INPUT.rglob('jeet-voice') if p.is_dir()]
assert len(roots) == 1, f'Attach {DATASET_ID} using Add Input. Found {len(roots)} matching dataset folders.'
attached = roots[0]
WORK = Path('/kaggle/working/jeet_adaptation')
assert not WORK.exists(), 'Use a new session or a different working folder to preserve prior results.'
candidates = list(attached.rglob('train_indic_f5.py'))
if not candidates:
    archives = list(attached.rglob('jeet_kaggle_bundle.zip'))
    assert len(archives) == 1, 'This dataset needs the prepared bundle: code, train/validation JSONL, and WAVs. Upload jeet_kaggle_bundle.zip to this dataset.'
    unpacked = WORK.parent / 'jeet_bundle_unpacked'
    assert not unpacked.exists(), 'Use a fresh session; the extraction folder already exists.'
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            (unpacked / member.filename).resolve().relative_to(unpacked.resolve())
        archive.extractall(unpacked)
    candidates = list(unpacked.rglob('train_indic_f5.py'))
assert len(candidates) == 1, 'The dataset must contain exactly one training-code bundle.'
source = candidates[0].parents[1]
for required in ['requirements-tts.txt', 'dataset/train.jsonl', 'dataset/validation.jsonl', 'dataset/dataset.json']:
    assert (source / required).is_file(), f'Missing bundle file: {required}'
shutil.copytree(source, WORK)
os.chdir(WORK)
print('Dataset:', DATASET_ID, 'mounted at', attached)
print(json.loads((WORK / 'dataset/dataset.json').read_text()))

# Apply the current tested code without changing the attached recording dataset.
import base64, io
CODE_UPDATE_B64 = 'UEsDBBQAAAAIAGOzR13pBe+sZAEAAOkBAAAUAAAAcmVxdWlyZW1lbnRzLXR0cy50eHRVUcFq3DAQvfsrBnJpSCNbXsfbbeMl6aGwUJqF5tKTmUhjWyBLRhqn3b/vOKWFIhjEG97Me2+u4BQyo/fw9O3rD3AB1CuF11vmDO/OF55igJ3SVSlFX38CGyFEBvRMCXgi+P78DEJwKYaZAqviCs4uBLLweGo+T5iQIcc1GQL6tcRMGXxE28/RkgcMVnYOlPoXZDP1S4qGclbFcNdvEh5gdHwzMS/5Y1nKf1pflIlz+W94eQrWmS93SpoPejfsTWPbem+MrrTWSPhhoIOpqt2haWuD8qzZF2Gdl0vXaVW3qik4YchDTDOl3HWNag6qEh9P3opJi4yZREsiT7jpH9B54AhuXmKSLEZ0kiFsjlKA8+UxpfhTFX+Jx26nWlW9v2+KaR1HF8YBDfXi5NhVmwBpaVnIMZnp2NWq3pBatX8QXK2L/8ESZ7CD87Txda10kY1bLkexo7dBy9vZbm1kOcyGVgL+BlBLAwQUAAAACABdv0hdmrkt0Z8CAABoBgAAGQAAAHNjcmlwdHMvdHJhaW5faW5kaWNfZjUucHmdVEtv1DAQvudXjHJKpE1oEb0gBYlLEQLRCsEJIcu1Z3dNE9v4sdX+e8Zx0qRoyYFcMp7HN+OZz1OW5a3S2ISoEfZK8x4+aqnE7Q30/IzOg9HAwTq03KEEb5E/ogPJA/cYoPrED4ce4cP9d3AozDCglijrtizLotg7MwBj+xiiQ8ZADda4AFxrE3hQRvuimHXuQBk8zudf3uhZ9mefoSwPx149zDj3dCyKr3d336AbDxXlUj1lqluH3vQnrOo2Fa6D/3H9syCgNmG0Snt0obragQ+uSgh1PZUrTUB9mlP0hkuWVZPd8+F05r4djMTetyH4lktu833msOC40mxq1j/jVOo029+8SJaVo1dRFBL3MBBWVUPzDr4YjW8LoI/a+5m8wSqtaSwPNAx4QnU4Bk/tlfDkVECSYKyNPMQRxaM1Sgd4UuFoIrUcTkZQHjcOK6GOE3DUzHka7Xt3iDTScD9aKoleOGXTVTtGfRHU6lUkdUIyPoVUZdMkmjTOmFDuIJwtdmlKO2LK76iITt03F3ETgOq0MTRSuf9FQGvE0c/RdP8dUFN57EP35mozskfutNKHxvGAM8CehrSCuMbmZhOECxGH2I/suFzEdg3ch+ahN+Lx8hVeb0YT1xVqgRdDt+v2iPJyvdspJZ7UmFAcDQm+q0oRJSdFKWws6wUo6zMYIXii3YQ5/hKqr7J59QrHxwqvoGzpMEXTckrPp3v57KrR9hy+eldVvXu2pSxtoilLNP1Ln9nHiH2LIV+wy3GjvNgy17Ity4ttZhNLbMouL1Sr1CvKZMe1ZoVI3GCZGxPeoli8Zg5kl/m02NOgsy1JWT9N2NHgq7SIWxkH66vc5x1QK2naxD3amYXa04bXfEj7veugZCztK8bKvKjy8ir+AFBLAwQUAAAACABdv0hdEi0EyOUIAADDFgAAGAAAAHNjcmlwdHMvdGVzdF9pbmRpY19mNS5weY1YW2scyRV+n19RaRD0QE/L8l5YRnSCbMtZEdsyHtlJUERT010tVdS3raoeaSwE2Rf7IYQ8OSYXCIHd0ZKYtclDSCDEf6V/Ss6p6p6+zEhagT1d1edW5/KdU21Z1k9ZygRVjNxj6TGNOREsYoKlARsFWRpyxbOUhUTmjAUn2yTNSMRTNlJFytNjwiXJmYgykbDQtSxrMIhElhDfjwpVCOb7hCd5JhShaZopisLkYFDvieOcCsnq9a9lltbPci6NqBBsUzxhtaB67RD8/yUYZ+hyqk5iPq3JnsJyMHi2v39APL2wwSYeg0VDVzCZxTNmD11Qz1IlD7eOBqDQRRkuTyUTyr7jEKmEjRKGw+pYkiazOZUu2EDdWcYD5uOjZKpW+wI3H5i9Lk+ShSyWrlISNIQ88KPPaqbH+w92H/nPdl/sTfb2nzhkD98//OweDU5ZGi7Xk3mqTpjkL5kYDO7tTHYf7T3Z9Q92f3EwgTMeDgj8WeXiVbn4CykX35WLb8qrr+EBFr8vF2/KxV/LxUdYvC0X/yyvflcu/m7IPpaL9+Xiv+XVa6J5flsuvi8X/ysXH2D973LxR0381rwHMd/CS3h6h2KQHrf/pBlel99+YzktQ97q7e/Lq98AzQfN81H/X0l/pVV9MALfGPPAogUwGIIrQ2qU1Ke4+lqrv3pdLv6GhiNJZcd7vXj3k8aKN0b5Ky34O32UhWZF8X/Qxnz40fqDw9M/al+9N/KNwR81Kzr33fLpX2b7P0tnlos/42m1R44Gg0HIIpJQntpDMvox4akaGxMta8JiFmCJEBoELFdQbkHMc8LO85gHXMVzkgmSi2zGQ0Yo+fnOCyAOyYwJHnGgVoKmMhA8V7oGUSyUJOYvowmoIjZmt1RhViiHVM9MiKExAf94RE6opApS3rA5xBIMECDix1DIVosU/wyN26KwATKyEDDBswoVjb6whppBF7iA/KyL3d0Rx0UCVfdUv7FDZiwHZPB8P8wCqFDNKbVXYBuYjRSXhqGfALDQOJ777DyIC8lnzD8WWZHbgn1VcMFC70AUrCdCc9JKsW2NRujfEQ8th5ywOPes3crVTQQSmvKISaVD4QPpbSKXuAlC1TxnHqJOLf9xBsC59Tk5/fJlA7AYyI6bVmRW6NKVCIlEi1h5Gtw2iUUlkMhNWoQ82+ygUiW8SY8f6Mo+2/VnHSl2rlpepAEk2DIvm5OeZSKUK+bcIneEgL3Om88PHo6+WJf+N3uzY+sBvAHcxr1tkiVcYdBFkUoCIAv/BGMjCYza/Cl4M4a2d0u02Izr8AcnGAXp2RYtVAYbVpAX+qcIqTVsQmje3ygUOEcRRGkK3QBEUJ18niVVBu1VQaovT/SMKTEn958/2CH7+48JFM79p89vFi4Vy2XtYYCkxrJP7t7ICECSF2oUctGOz/UsS/DAE52w4DTPQN262O6E1NQf9ktX0ohBCGQmJNHttDV8GKAzOkGRbJJb/6ByWaWybFonUD3BocFk4ryBNTNtZKBtVjfmOKOhb7YGS7rWpl2XoAuLytMVmKJyt4KOLnTWL5e57mMO4iQFE5K2DdF+DYmeX9p0Xbkt1wO0Z6IFc6SQTBKuJNFwjcNag2/t5jHsW4qqbG1LPejAaWtWFwe22Bq6XGrb7OFtFlWTUaMc5UdZkYbbBHuJCHVjMyAMnU63QTBYrthWm+N1Rq6OqV2GSrxHUnClbYvsTPdI/IUGWbG4hkri0eHNoVWD/xHxvE5MoYgxBCsOq9RAlH5IhO7j+fYeNG5AW5YtyMiCTJf90zfAWlnVRMcwHVq6Hfg40VpHfV6Tcd6StgWgDS1Mqz37V7QuN65TYK+c/7q8hj0oK1z2B4mR5Me941e+vk7WCi0eZR11h3Bt7i/PAqnSXsMABe66NeFX3uKf9bSa5Pr9DpWs7YHkjKuTpt/pZupaK8KbExh49gGe61C1dkBLjVtmV27CrcSCdX23ctPszK6vV26hgiGeN8Kd7pGsjV9uJBuhv/HlxuONib8RNUZ10LDR7rJzLhXgsin0dG633nHFIN+FPez59WafWvtaAgFOGM0y6IFJAdAyZVDqZ3haluRqvk20Zry0wv0PWpwEr8axjiqQZuDdM4FTQNrzbHMQjAd2mUPtUVwdLdNwBcN1znVvaU2rocZOj8Q0mYaUmMlh3Lv52WbbqYBn2TYbi7p9bfWeWOFhR0xe+PU84ZCp0eRXFnnVb6Oh9hWcuoEGBE64xLJzh5hzp4Sl0OjxO4KtveSQrX4MBVhuR/X3BozDhZZxuXkRs9SwDS/H5AKfLlcAD80AK1ondptnze00AOUsC7U6tR50hr2kUidSgyCKhkljxlq56JDIqqc+3xg6vnM3XG+XdGmeY8BWcvNiPQLoSXSsneesp6hmyXFtXhXD9cQG6+9+egr0+liHd47clCY302993tBv3UifM3rqy4DGLYOavVWmy2sKSDA9Vnk9t1h61gPRFuWfTk8opMdmlctWVzhcSWdcwgAMxL2vJl261ow51t9xeiXUfAMaNtPa8q0pXizjFfX1Hc9IXa5b8q5j8VtddtzkZ5faDOTjVtKuiNMJt4xD/30MGMdwOvbRVewMvYqpCTtj/OjA4pgf8ymPuZo7BLpQqvQTBADuf2KO9aI4woBs+b4JaKtEsIHAhUlPgTAEInqattotA3zthkWSS9vE3wGwkPhxkMqAc+8hBWc7Gk9S5d0d4uvul4Q1PaVGkwlULVwAM3LR2HXpkkfaC7g/zaBzIi7JbaLvxkTyhMcUjJ3XeD2Hwaluru1JSzBViJTc0RtwR8aZ1N6f7GIbcsizIsWGWK1e0Lion/f07UEvoM1JZB2vWH5wMIH6hz4VUejwIeAekF3CjQgbvtf6RtM3Z6ttzs/YfJpREe5BZIUoctVXZP0qnagM0Cl0yX1McH29Nc4CHI9BhYZz6MRwOUjymOHkWV985Rp3bH1iHAI3Mfx40PmE1O5H115U2hAexJmEohkMBsDt+4hBvo+ztuX7+LHM9y3DLCiHkpzMIa7J7jlXtvmUNhz8H1BLAwQUAAAACACxACpdPzIfo5YAAACvAAAAEwAAAHNhbXZ5YXMvX19pbml0X18ucHkljDELwjAQRvf8iuNmja2CQ8HJRQcXBReRENNrPUwvJU0K+uuNuDze44MPES92mN92auCUfeIhtNb/1bP0ucQ1sCM4SiLvuSdxpJU6U0fx52Cje3Iil3KkBsb88OxWU0p6lF7trQRhV15sbjk0UG9fhw8MQQJ0Pti0WQML3Ja1rhZQcNcKEZUyZqY4cRBjYAdY6bKh+gJQSwMEFAAAAAgAsQAqXZdo5oI1AAAANwAAABoAAABzYW12eWFzL21vZGVscy9fX2luaXRfXy5weVNSUvLNT0nNUcjNTynNSS1W0EjNSwYKFBXrKBQU5WelJpfkg9hJicnZSfl5qcWaekpKSlwAUEsDBBQAAAAIAFeyR1388m23SAAAAEwAAAAeAAAAc2Ftdnlhcy9tb2RlbHMvdHRzL19faW5pdF9fLnB5DcaxDYAwDATAnilerhGThIoJQvyICHAsHAq2h6tORBYny454re+MGlhzOWgaI6op/S+to21w3oXeazNkU6Q042r6nIxJRIYPUEsDBBQAAAAIALYBSV05X+tl6w8AAJsuAAAeAAAAc2Ftdnlhcy9tb2RlbHMvdHRzL2luZGljX2Y1LnB5vVptb9tIkv6uX9Gr/XDUnExLju1JPOPFOo59Y8wkNmJPBosgIFpkUyJMkVw2aVuZC3D/4f7h/pJ9qpovTVJKPAfM6YNN9kt1dXW9PFXN8Xh8maefVSJeq2Qp40jkKlS5Sny156dJEBVRmqhAXOHRvzwSepMUK6Uj7Y5GF09ZHPlRIfyV8u+zNEoKEacyiJKlkA9pFGgQW6cFUQrUfqAeIl+JLJa+WiuM1WWe5ZFWoHW3UgLUZe6vokL5RZkrIZOgIQdWHjAFvIgwjeP0UZxdHb5eyVwW/0GrZKmOijTfuKPbMsvSvNBiIbUSjyparvBiaPkyjjdCBjIrsKWWay0eo2IlsDGh5VqJB4xclLEkeuPxeDQK83QtPC8siTHPE9Ga1gDVJC0kcaVHo6pt6ddPqa6fHmWeYBfa0PGxAWyRZrly4dfEzsGcXMTKDApkIf1YakinHtA0mRGZLFZxtKh7b/BqOopNRiKr2u/+cXPhnf90cf7z1bv/moqbPC1ScNDwm5TrDDLRIsnqJp2WSRBGsaJmHRqq2o+yjaujZSLjmnauIK4sVl6WxhtQDLurnYwEfoanFCfbsKQSneZTkSSj0dvrNxe/eO8vPlzdXl2/E6diHC6Ojw7kq2N1NHs5ezWfHR69fDULpDpaLF69CtXRfB5Kf+GH49GH63PMft+ZPQvV4kUYBK8OXi384Pj48JWaqe/9I3n88qUffh8Es/mBCg/m49G7s7urDxfe7dnbm18uvPdndxeYf3DozWaz0WgUqJCVz4tI8b01FDh2Wo05EbrIxX+z1PHvHWwEs+nfROz9DTtz36ZBGSsjAujQL6DFCoazIXs6v3wroMznN7+y7qUltIn0DuvkP4gMklX5gxJFLqOETzMpVI6Nw1hIIxvBhkdeUWg3SmC0bllEsfb4uZY174G5H8zh1nrcm+iuHbAql0ssSst5q7JRsVVIb16QPiZEth2vZagKPlTtdk6aVydNGvHgIr2HozmFZbhLmvDgjH+69O6uf754N56INO91nJ1f3N7W3UzAT9frlCj8Piab96JgfCLGMjpcsCvYr7zUeCrQ/xBp2BgGdHUMfcwHOvj/F6bMzsFjn9HhD+rx4R9nt97VuzdX55dH3pur9xUvUWhNMudMP/YdoAH9cEg7nHbQROyLMfe7xVNRkaFf7aXMrFbLJrSI5VxVDPZ2EebjdK2jsBYAGXgq1laHGZi4keaDcVjuTWfFidXd7ox+UEewcImed2lxSX7iIs/T3IF+g5kmSpCr0grORSkEgWbL7IYHfLoVo7S7oRx7SucQVwmc9OnY8t/7rVSn4rvvjJpsk6/T2c0zpL17+aG8ty1tnoypnVrW2DICy5s2L7+Pg2gNzZzPDg6hqIHKihVeDw7wslIy0NR3jJcw9NZlXFAfKbR6Kjwz82hODRQvvVhuVE4zDr+0K6xV7OlM+R7CBHZBLgecd8XOR3/Kj22PCd8Qe1ZW483eNCKgqvfGOlOJe9qZUp3xWpL1NkTv1cYljPCg4PLC6MkZY4SRENSi1+WlebSkTnSdiAcZl6ohFEKLQWxqmuEuDV8u0MRaOx1LoDXRCYxAnre7Io/7Uts3mQV6W6U06v++BBJZq0rza523NGdF4TQV6glyJqTRrFBrYr2SWZYlx+ziDP3CwfAp6SaeT+/yUpmxuQL4SMwUBChGAsK7LOP4Jlc++7oPJn40QednpTJBjZoBVBqXjDr2wRgiFTbB2GX/8vJOIADpKFA4MiYhZAmYIHXB0WZkzj8EBEI0KjzP0SoOp3W8OmnjHUc/CoOtzGioW43E2VdPLU2zYkUR2nlS4QMmZR5bYuDmDY8XzusprGQ2FXcTmiVCJQmcacIsISRavDigoJnywDtPlkGUTprYyQdsYhQHrFHfVXIrvGBIG1YOVpi4AGjb/eF2hahFmeUQjQ81SNJkz9DrcOzakYBxKK8NZOXWp0Dru8aaXLLbqYATAlYMTi8lvFSfKaMptuDdSspEh6XjTCaNEt3WmP619BEQA6cGiZNGk84SCBPeCL3QG4Bo0lniQGgAhVjtlQXwiUTW0GYIQOBMrqtCS5VgYFEfOHkuBlPTNvE44Vg0JfCTaD+PsnqELlSmTwT7akJZmZsEMs/lpqMg72tDgQk6d9OJgLtTwCSyALwT9z997miB67otc36casPZFkVm0rEiiGCMGWeXljkgmVgo+B9sHceHBAZwzZyVHq5UibzSkUrgjZirVCyIwrAkgzYiAO01mSSCJxvzlKUPhcIAmZCbMWlYpSBfs1gzpBLns+FsVw7Px521rXdmPhNcdv1HlTueVhvodtbRldjc5XY6fUW+6VpMh8xXYH9rqOrJR2wWF/yPTgpuB21dsvAl48PZi7EJR7mDAQy30DgfNi5hFUGnGZHhUeV9n7PD7wzGsE7Vzkj6UEekkpQs/7OMchWccBv20KT5mkxkvJ3Oqihgevv71um5fro/gN6spaAYRkvK4Wt47w7JTow6YJ9Dj9q0/FXclAtodx02RK0fWqxLXbCrjpKVyiPKnxBD4ZF8iQMLxPVZyb4UBuba8IbIeIbDLfCy50g5xyCoiT36Rb7PcGkPXnTv4PB+9dkCTvSzsCnTdzdyHffG1HnJaT977Q5jvo13bzta9fuT2M42HHsqOLSIkj+F+Z5p2q6iu5H6tAxzfahKP6QpnAYxTup2mfSIqiSnTcrUPf6Jm0kEnGLSnTgEul32CWquGe2QrTqt75h2dtazWjPDVcCnFhy1eiq71N4yhzwcE9i3DSxSx7A4+f+MqjVI+KwERQdaAQEe4W5aVyso7lGFAylQFiMpapfeb9fcjcAa8T4nvPCLx+t7wFfk29qIwVhvKqraVG7SEx1CwrCUhikEQ5MCVWgR6ZuMH+VGeweBBbvp9ygfKMCvQcZgM+LR48qZw4u5d3CGeJlMLM/1+nJ+3FRyNIumBt7wcslScXoMT7XYcGdIZc0KNLri8mZ+bBFDGpTzAB8Rv0KUkAK0KQq6+Fc9AEEwjpQNEK5RYEOw1MqDeEgudng9ReZYBnLMXtxsld6pGrAI58eeNsVVFTg9yOr07J/Bc1JJmt2JM+n7iAHIHYQIw8x00G4OzlBY8B6REw9G1RC52mt3gMXNAD1rJNYwHOjNVHgQ0RZlG/Lq1ErSUbzJkK3WFsR/irHYsr2PZFyfhu2Wpxn0bU0EnY4/2iJH4+ksFRiOwd498hCn7CZ6QrSzJrazvwBpDeqq30yZwvFFnSofHM5mM/HT5zp5oMQ0K3EcS0T632mNL3a+VKU65syej+Pf5GkmFikUt0bylao2UL6yiBA5H3HScVv/N9C59N2q8m+XI3bloF3jkw8y4suBPha0hql1Vmw8Bj8O5XZ/b+4LnJDTisqpmQTk7u72miXb5B7vZBE9mFsX+Jg0iaig1yZRtSvhyKAp8n3cmyMF/9TmGwlTMKn2iRVHuJNbvfnx/aDHykn4PVPy3tNYHW28anuuGjZWhTijF14Q5XbyMhWEFbiFT74oYYkfTQ/9/dRRg99yysVZD0izdFOKJ2/7iE4Kaeop0vxgVuzmdFW1gBYlDM8gg14mLjfBHOou/k94YexSedqlGtYWu/hA5auqkGAOyMxktLvA8dD9GcjUmI2i97pjESn4OTWctDLq9rvre7Q5Bv1og53MPr30vhf7CEBx5ZQJ74tw/Dut+8UDknTh87CZfg8OmXs67kEmG4doubyOdiaMo6iFpMKr7Kw1X/CUrlRkTOF8Y9jWPyCZTWHzkE+iHqujQh4Nh0jXgvvEmC2leu1pDRbYe4GRz1HGbCLJNt7TVuoK4DWqPMEgc2MEQzim/5PeHjg48rbTTCXO+GkxnlCquIKZxWqY1wGlkN4pxwzosIfVy4WBLDfnbyF/iJ4CjixOx7+dfRh6Rd5Hv+TQ4ri2TviL/LwxYG6PsF1bwWH206zYg2TOf31zJq6v3zY+Ec6H7q921Bs6PrFfRTYmC7RBsd+KgIDcXk3+BIaZkoft5RFVYMA4PtqT5uL040cQ/TQdlLM+gUa33lKVr4ehIQprx28SS1ir4VCYbKBCJN+03DeGSGO0oDGlzU3JE7C3HvfSoa8WOGyxYIj92h1YCwWDemLqjat6TwbC6lZ//nhljIBsVZ2yy3+WgDsMUEGCRN2lN+DTNctPdo8YMKwbPd+lim2SZBlOkyxZMcWqG3VTp5Zwk0KBjxcHln714mwlrV8hKYSYKIwgp2ZR8ZjmgSazqtM5rgRSJyCCzJeqMEPcFirQhxMtgTZKcOBG/nEPKAXnAP/0r//53/kRREDlHo0zSpauuCqqA2gtMI6yDEyRysKzRPzJxBQ0fVny8UrioYyDOvkgLunQrRTP3mk/UpLQXbrbyNqbR2tq1fMt+7ozwuAklADLFllaMBthrql6ud2oxAcnfhRz4sW8/I2uFb7FwC0PrcW9UMWjQtY1Z2YwvQtRa5aqqNw0WJwkYWpyVHraOiLkQa5x+AQs2OcT29xuco4ag5tg1HT6iCaJijV3fWtr73drUwXNsXD1VQcrnw+VRG45vG6eix9PDQNBmfNVEzXMj74pXEUoGcs2lYSO8loS5ftkWdXc7SJDR/x18PaeVQYYbANwtbkHaqhVt0Gswpm7lk8O/smFtkZMSJX+uLSrGyKm0YHZHcbMJQ7W1P/M+f9ayaRd3EU+jV1RB488PpyI774TBxNw9aOYubPZH+CMXISS+Z4mzFn8QG6UzwfCWdG9BJeRYXyFvFd9++pVFzia9orkW1Mg+nUDY1WZ+FpuZK7sOVTbWChHWpSVNZKwvPlmUK4fBKdhYGo4s0JPJ/g6Ft/diGWwZDO+Dm5N8c52jlblbrrFQ1Zlu8GFhJ1Xb72TSFMqY405ywnFWq3B8dZLh75oyBQcmk1Gt7tqNAAsW24vQO2rhIYz+CSfeedBvzGj1c4mXfGeVMFg2r09sLjXYCrIam+v4gIdW24r6LfjxqLhrVdGavWOMiqrlmf4qG/cTV2QD28Rp2BGp6YcCLUmo8qlr5jJ3EAsAITqu5lO9aVerpd/VF8euvTgbJHKD2Ym57fmczRnLTfkhzT0YEK5aiXx3wypLRhsGzjbWu1gXnea+/OrHvTbWfkYsGd5ENsxNP077dj6bqX+/ZkW3NCmSKK5PuKYtm5I4iaXoxaZDIWnpJgfn2xntBreiwcvDuibsRcH3x+/dGetD4ntkIdsMzCTrBWnoiZBuvCtGFInFRzbqqy0nizMtzKMUcC+CXh2+Pj2FraIBXnemmGOqblwG19aQFKzGnDaEb2S8O6PO7btx2xFBW0FvqmTDbbxV3FNOfUKKBSwuoDxlgYLda8GDB/7bektVg9cmYzNB8arKGsNvh112v0It9rOlL4KezH5msi4wIbpBF3m7mzaIgoLzFTCmUy2dzdsYMSg+tBkPhUV6BuvObW4b5rsErRpm4z+DVBLAwQUAAAACAADAEldMAvx4PwLAACVIQAAIAAAAHNhbXZ5YXMvbW9kZWxzL3R0cy9hZGFwdGF0aW9uLnB5zVlbb9y4FX73r2BVoJU2Y8V2LiicnQXSxtkGaC5IstuHgSHQEsfDtW4rUrYnrv97z4WUKM04yfapRmKPqMPDw3P5zkdOFEUfZGe1LMWbutD562dCFrK10uqmFr3R9aXQ1ogaBq6VyBsQwlcgvy6bm8NK2nyDQmVjTBpF0cHBumsqkWXr3vadyjKhq7bprJB13bBac3DgxjbSbEp94R9/M03tP3eyLpqKlbXSopjX9AEe+YXdtmQgj7+st4Pmuq/arZBgeOuHTNPXxVqXCofN2g/bpsudOpPrdpsafYnb83YoI6u2VFnblFu3Ks7w7z+r2jTdQtR18DLtrS5NWkgrvdwr+GyUde4BnddbadKqKRQIWmtSje7P1s/8hLfvX539K/t49uubT2/evzs4OMhLaYz41Cp5pTqnLnZ/k9MDAT/g/49KFgIVyAvY6fFzcfXPL6Jq6gZ2kjddYV7AB4hM7UJ6SCaIk6ckGB8vPidC9oVuOJiotVBriKeutc2y2KhyvRBd09hTYWwn/kPhWAjTlpqHEnH4k3jX1Iptwh+clOIcsSTxGD8nKfi2Ka9VnAyCes2KBOSK0LWII9tJXUcLEV3LUheUP1EyasYfkDBK/CrLXp11XdPF0S/1Vd3c1KwrjZK5ITcGDFlNlGDupWUjCxOXulbJ5OW66QSOkknjZh6LdXRHa9ynOL+McE+yyKy6tbGq86aA9FxGvV0f/g3ekSjqMfFUP2wbh1Pwnm6Dd+ehY9Alg/3f8sAn8qI2QlWt3YYuwL2ABtzKA9qw3MA/k42C1CqivMjwdXS+L3p+MrwqKbky24xadrbsqj81G3ny7HnsZoL7LrYWXZSkG3Vb6EtlbJyIPy3ZCJaOzqc27/fCSzRY5BuVX5m+EpU2hFehOwLnknoMXXTuI/E9i7zVhmASMrU2OUyjjAsqp1S1KxwqDV3bUa2rRRCJh2hMJl8qq62qhsoDmFC3p6iElNkesGnlQQisDvyCUV6OQV7R1DGlKJpQytIqFFuT778e9IUoAHLVMgLsl/bJSTSpXNIEYTp+fnR0JCDNGEcA1yoaxiH0c92m2qwRT1RMIkkqy3Lu7F1Hn922KreqEDyXUc0hHIB7lyuPXMl0k7C9CYzHbutPFuIE1ja4pxiscrtK5tFhTEfgzqiveKv72vzeK/VFxUfJIkwfQGuMHnTLtb7ELkgohoiclXKrOhMT6p5C10jfNkVfqoUAdLfZRdlAqlJ4wegTCnGpjV2BIPRpWSmrfIgBn193uLqwGyX+8frtC6FoDXoGH0ETe6U/C9YJ/bcQTW/b3oLjYEmC0sdt1/wGToWPhPeol/IYUKJSHdhAhqbBGMk4nctQOBRyGzkIqutY/LgMN0mPkPb8FAR/N/Cfvfd4XfBrD/6pemPFBYwpe6NULaDj0Ca5oRWqtUOh8yY69XuvIRGyyw5AJn4tS+NwHjGRnEIIv/qBbVodhjGB5A83W8OfDPw5HUV34mhQhax3vvjnrndruxxbtT68ZM34BAax+cMQQCM6dXie6vbJp26RBWQEfm0DCbUn57CqwzY+a92QES9tU+kcynMLnAUI4LovS/A0NcMbpS83wA2ZJWLS6XqtOuh7SmAnVd1fjTh7+/LwSm2xFq5VPUk0x4TWsAeEL6BBIbHC5TKka4yGrikRe8DP7DxARhCW3RZeUf+40XaTmX691rdxlNqqdRngbV2Ku7ETRqqSGTv3Dmy8j07FNaZdWigr802cpHnb4+8GDL/sm97gQwn+iaf9FCYveOoYLgNkV2XA6aB5pQjhvunf0+9he7EzjdA7HjYEgALBlUghl3cRy2QIVGBkxAFAVoR5Jy2OOfoe3c88A9nRljJXzmucHCOZypC470mNgonl6Yxwwgt1rXM1Mj0CzSFffjEIPbcA0uMSjssD7TEC4wP1AT1Z1pc+bWoATUEQiSMf3/2MypWshkxptzCrzjp4u3TaUmiM5GLnVTaLeB3Ddd4XEn51kI8247dxco6Fww9iuRQRCkUCKLgSq3OHftuxdtFah/5uVfD3FdoRu/WW7i+Ei+Xqhqpwt5vRdKNUER+fPHk6JSA8tZJ1L8vsARnOKgVunTEuDCBv/HwyvvYdGExDB0Nm+qDusBrUsRAZ/POQzz0OStLtFDa4QjXnyc5kB/C8iaGzo8o9/ImdgQj/EUBcVx7jAXUOXV8PModPlbtL8p5T2baqLmJKQV5wKuqwFbhfzDMSYDXYdNwTwxB2ynIbNiAXK5dgY/b58qFczQxXRjy2mLCCOCvB4dnOYYlecS/eHf9hEeQzv/aJym9U2+SbgSM8PeLRUskOyydDFnbKVQmvj9XhMxaQed5Xru0Pk93cPdRj4VFXI577YacLU3TQAaIEBAh1KzB3gQfxkaP8HRn3ITbn1l00jBcML1yXgh5uLDVvNEWMPYtpC0ZLbIAGNQBnHhMg69gT4kemluEG/Zi33z9PvIQM5OibvAMhiQiHs8VpYLbrOUjbGI0HHp+r4VmXA514o/FVqm5hN9jHUaWst3Qgxi7RFbqDk8/cqtfQKM5ojjPtvWNybv1a3eD2Joc9SlIwYnZjMKQkdA8+XLM4Vd3XxcMjuN/OXedJL58sEWZILZ057sVf9kqQJpb4ZgAeB3gwnrFEc626UrZ+t9dNLi8GvkpPwH1kl1WyJYHe3QnAUQcyDiB2WPcuJ8P44gFP+GT/go1MRqP5NoFuD3Asx7Fxb+CL3F9b0PLc5wdHufW/st0peuPm3V4NsieL0cQdyRw5oJAXBjobkyhsoq2ua2y7uDRUAbQxcefWvI8G1S49g26Ev1zC7HSh8d1IPsGB3zrXTA4zIQMfuwlDYAsNQH+hcwYvTiPpy0JW/47HJUFft5yULs+HEx3MnLdzhk+kSoPaC4JDOCpCHHASVl3AErTJLtbHz4E3ti3nRsKcwB38eDIjH3DhwF5Qlv4M7f4TDccOpt05rFj6tZw1y6nCESrS6grrHnYMITVLPByAEqz3rLlajmcFh4FjryfcXO6ncy6BPWNziQgdFjCYxGAizqfxP4sPcIBQ3TWfHi+g9t1Biro7Qy1S8w7qDsBYlcUh8jhq0NyW9p83+K4Q2m6Ea6UB3XeFCz22xEgduSfVmuEJy8ytTCzyUsXHCw/7j8RxMjlm8TlV1wE/arqCooVn6JgVYPsnsSQgC64gPtIfSntQT+skqdnAcQJ4Oukap+yjXQQjFtrcaO/RgggHT15MmtSMHV12TY/5TKIrVnPq1D2aTJzyvKGKUvjv2Ccgd2abrIa2G6RPaCZdBqGZtOwuTwuZ45IRfX6B5H8Cloz1IHsLGAT+9tXAd0b0e1IaD7DD/ysuOpxKUMWLgJmIG/xyAZbqW6g8VfwRljqcL5PdOQwwKf0hYcdYKUxJkl7I/OpGdsXsEOCm9TV9yOIhKZJZjskiw6uLAcGAsPKXFXmp22x4n03g9zg92ntjOnPzMPvBe9MH/YxTNYLfH/Gx9xVgxkMb9n4hBXOfEdY8ApI8DPMZ/o8AKnmDYbHgiT2yzQFYnbe8yNQxI3z3amZaCIr+53+FWJoL/ex0zxqT7XuO7TJ2In63E9KIADI6ZaBc7L4nB0en7Og97/kYhQ5GoYcOaXsmzsLj7232SHq/g4j/OJW6n1Ej/Gk79C19I1T0VWti55fV4fE5YM+67M1mBquxD4F3Ic6OkvSmg+zm74N29fFtfm2XJwkC4+zrojB9OFY/LccD2TSPOuDs39eFkZntSZFO0TksvB6LkATw9RjeL0n99ALZp33sb5oWM9EOysLg4eB09uVlIMj39Jm7jEcqB9L0hUEgRGvOZE6eTmU4dxC0MPxjUw9EgiwJ5ZjWh8spON9jhkT+hAqgpA5tj8Af7jJgtahrfJrbRYx4BFCX3W1a9xV0Mz5StNh8R5nQoAfqJvLkbTf9Q1oXhgXwYI80DodqXWFkeKTKMJROBnDsAcV77yNFDETx7O1LIUGPxNu9JPQesirclApLMJrQeopQ8BzIhfwHxMLHUBsUluLpkIzqhoIKUAYjL+b3SqJoFLeXSkkDpxk4O+FBotRXCu8qne0MEEN9e0KQccV8vc5Z5jvK3F1UsfzBfwFQSwMEFAAAAAgAg7FHXSYNV3gwAAAALwAAABgAAABzYW12eWFzL2RhdGEvX19pbml0X18ucHlTUlJySSxJLE4tUUjOz8lJTS7JzM9TSMxLUchNLUlMAUopZKTmFKQWFespKSlxAQBQSwMEFAAAAAgAjrNHXZfxAOnbCQAA2hsAAB0AAABzYW12eWFzL2RhdGEvdm9pY2VfZGF0YXNldC5weZVZ7XLbuBX9r6dAuX+oVGbsNJvZKKvMehrv7HY3WY/jTafjejgwCdpYU6ACgJYV1zN9h75hn6TnAgQ/JNpO9UOmgIt7Dy7uxwEdRdGJMPWSX5SCGakuS7G31tIKzW4qmQmWc8uNsOYNW/MbUVR6aRjXgsWnsylbVqpi3LKDV+z6py9JFEWTSaGrJUvTora1FmnK5HJVacu4UpXlVlbKTCbN2B+mUuG5MuGprmXutZDtrOTGCBPUtENeYsXtVSkvwuwxfvoJu1lhM2H8UG1ao6perjaMG6ZWYchUtcoLCQ9g2BSTycfD98e/HqUnh6dHbIHdpfv7+5PJ5IfWegwjX4RanOpaTCduiH0ifx3D+MrOJwwfuOOQGaGsUHAkVzkz9eWlMFbkLBelvBF684ZZcWvZsjb44ja7YusrOHQNIFpklc5F7t1KCmU+Z8Zq90zLul9BnR+ZTHJRsLLiebpyeExMjnKT7F/OS1O295aV0tizHuzzFveJ4Dnj7PfTH/e+Y3/7+NsHnLnmG1YVrFbycy3YncxnDsSsNX7Pqos/RGaNQ0yadLU2cCCdc0JwTEy2HZhpomEjJQ0x/FPlOK5FVNti77s9Iy+j6dRvuWCIGyYRmsZy+DEmnTMHfcoq7WZpyEN3Rrk0gn3iZS2OtK50HPndMXfCztNZpSyXCjtUlRKY2/Q2mUTedOM64D87dwNGCAUfCnsGP55jHI+xl0VeEAoGnUMwo/jhMpnZaSc1DvuInNrA8LgvKIwaLwMmY99A+2c+Z6cn/9jff9nqK6Qoc4cc1pJLwLwWm6lDiQdCGUcyj2YsIv/T33CG0fS8j52rTbyF/4YQziiUWv+7oQQjchV7K26E7HgkT261OaEupihbAqZ27+1hkSl1acJJdaeFLffiOX4WPw1t2t+v15LInATceT8OvIje1atSZtyKgODnd3N21+q570EkfQnP87id3YZvEr5aCRUk/LQWqKQqSDTJzetcVunnmpfSbuJQmucoaonKXRi7DKc4o2idsQL5Z5H8Lucpfrtkx35kTjvwMk61O4Cl4AZFnEkUJGmvqhrVE1Yg/sVVcjr/XKiKouOyTXqAwTEACDcOSIsOYY+yLBaYcpb+8qLNcYgAt1yyP6HckloaMPKLYIsF2w9xhoXSFFKhPZHSacLLMp4+lvmHbi9t7nQR5LX4BuZa2TD1Bb/GHhzKGFaX/Jb+8Avj7Ha1yQm+ZQfJ/tMo+BKRYutctHgQY2d7BzN2cB4MU3ft2TWftbcvuCLLcCl5MA4efPVyyp49Yy8CoqyUiJ88LTTP3AH190A6eptgb+HZ5PXr183aNdeKkqqrd9gf4fkeYvv7B90Gg2QI1uiD4BpFu6Q+N2fZlciu2aaqNVvKTFerKzjdJ7T0NSTsFgZ2EL9l+49YOq6MkURV3DpMzdHm1mJg6dJVdphD4uj2SPNa88YlbWw9Z71GHxC1gt+zb12Ah9845hePQPuttkbicO2V6PX5b//77/8cvEDqo+Xkra72uH1q30UUR9HchROKMbyOH/jG87aDMLE9BKmABrPh8R6VokdM3nke1yb9R1shs3mWiRXhtPwaJKtS5SZhvyN8yZENAdFUejIBRSs8N3wQpbTEVMdNqCilKWVVmsZGlMUMnbCyO5zjAzTPeyWxLBKSw7E4akDP0+E0jivNpXb9Nkg/py3fmGgoueRKFvD7tmgYT4iIlNGo+mR5jW9QEw3CZhyxmzFxi4hNq+uG5w1RO++YYCt1dKYZBCvonDKYINmOe3XlGQw1VOQmDqngDXaVODQm3m6mPobOur7dIesNUvMZUBb6UD8EH74QmhiVchVJYEAgTEU8tP4gXYumiaHKRgqADuVsCyGycDjgQIItDZghLZ/uiokSGXMjuvhYn0W++RGLjM53l/j+tWBxPwaCHqKdpipvRPzAwiRIprbqNHSrdpdRmWzWoD2lqK9VWcN9LT3yejFFBHT7+NqN7nSN99JQX232A11S3VCn7nziPDACSBW0f1Mk9BS79aOwaTqhzoyuj8Yb/f3wU+QNYdxQv3KBQHP9G1GQyK5wqROlcU37a3d1dLsCeUXFca3XXxwZGR5FSM3K2Wqr8B5zIRB+IwCoMA9L+ddieReUojHxSy1QAYnnPAzHmfaOSckz0fmIb5obyVlkNVcmA/O00XmgoF8L7efmrL0x5o4BunsqHwbomWIKgn8+TmMf80hgs21n8NpGzLV0dtvoUFTckiIW/yI2zsiMnYLCNI+d7Rk7qZWVy/Dr0MJhF8gj93tKN3NoGqkju5Q8+K5tB6603flCdw92DkXg5cy9KMDzVjl1xTO0dext2iusrtQ2Ir7E/4CNoy/aTVvwg+fgi169D9fGbgdoncfhymAYL6nAbtrVLZ3uGb7bOd7evbPflO4fwmYry8vUs5EeOEcSB8hOSbDjPlWxRRXG4Jl6GW8l50MAe/2x0ivUke3OSMV+6KsT4d8OKS0KoYlq5r40Preg9O5yb94wJW6IpIhSWEfFlpCxFQocoroEuRnCbhUt2N2wXey0ma4DPOz0bX8Ar8jjQXiRtsEAKaPBVlvgI/QOjhjJMJvozgHprps4puuGOoDuxq56+2tV9Pzuz7Ohaj6m5v3L86xXbByLm7GvuGYSjxkc2yG8733PVvUFzvbKcUsEYCH1sgmnN4CL7ep65aJtyTesFDDWBEfTQftnt3tXh+v6mffkxb3Jvd5VfSQHBy8HyK2dT8Yr+m5JPW1XsMy9BKWbn7uE9tX/35dm+jTXfyzceR3QCdFtAQ6BEL1YTejrZTxNrsRtKxIY04BuP2dFdNcsvqfRzvmUcJXmevPkmsQuV72Ec5zvbuCwqBFv7jV4mg3ne1k4h3qi/c+HuLYW9HrkfOS8tqS7ajrvImpcxr0ra6Xce6qhXPsSrRUKI9uCoUDOH7iMbsn3Kcf8EbkmACDTPHXzXXXaoeMgiu6Vf9yeq0v12cAQqvuFC8To+K/v04NX0Yx55riIdunSN+zoNitrQyQ1Q065rF5p1GbcrSi9cFFW1GAqOMfZRnfHJEoDmiGKQ4lCQf8qSAZqHTnztQB9TcXR7UXkmEGOLi8V95fhdheNkG6ETFXrTOwyiN7ixg9e0t14xgh/f0GBXV6N3CUqkxRmo7J4IEyvSKptlcbyS9eC+g2o6C6tXbAPkmn0KFsveaXBTTis7UvbDK1yTcRoEf1TeQeh0ubliIPo43qe63PurdWzfuOjm/76fHwdfbzexrfuxpfXy5WJ/XKCRm8ZU24yKRc/8tKIKfszc7BGdTb6HnL9wP1BdtTzjaQWq5JnOHbns9nw1j1cUeAo0cp299r4u1Zw6XW89Fe33TcHo+u7gH1qed/rjzLUaj35H1BLAQIUABQAAAAIAGOzR13pBe+sZAEAAOkBAAAUAAAAAAAAAAAAAAC2gQAAAAByZXF1aXJlbWVudHMtdHRzLnR4dFBLAQIUABQAAAAIAF2/SF2auS3RnwIAAGgGAAAZAAAAAAAAAAAAAAC2gZYBAABzY3JpcHRzL3RyYWluX2luZGljX2Y1LnB5UEsBAhQAFAAAAAgAXb9IXRItBMjlCAAAwxYAABgAAAAAAAAAAAAAALaBbAQAAHNjcmlwdHMvdGVzdF9pbmRpY19mNS5weVBLAQIUABQAAAAIALEAKl0/Mh+jlgAAAK8AAAATAAAAAAAAAAAAAAC2gYcNAABzYW12eWFzL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAsQAqXZdo5oI1AAAANwAAABoAAAAAAAAAAAAAALaBTg4AAHNhbXZ5YXMvbW9kZWxzL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAV7JHXfzybbdIAAAATAAAAB4AAAAAAAAAAAAAALaBuw4AAHNhbXZ5YXMvbW9kZWxzL3R0cy9fX2luaXRfXy5weVBLAQIUABQAAAAIALYBSV05X+tl6w8AAJsuAAAeAAAAAAAAAAAAAAC2gT8PAABzYW12eWFzL21vZGVscy90dHMvaW5kaWNfZjUucHlQSwECFAAUAAAACAADAEldMAvx4PwLAACVIQAAIAAAAAAAAAAAAAAAtoFmHwAAc2Ftdnlhcy9tb2RlbHMvdHRzL2FkYXB0YXRpb24ucHlQSwECFAAUAAAACACDsUddJg1XeDAAAAAvAAAAGAAAAAAAAAAAAAAAtoGgKwAAc2Ftdnlhcy9kYXRhL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAjrNHXZfxAOnbCQAA2hsAAB0AAAAAAAAAAAAAALaBBiwAAHNhbXZ5YXMvZGF0YS92b2ljZV9kYXRhc2V0LnB5UEsFBgAAAAAKAAoAzwIAABw2AAAAAA=='
with zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_UPDATE_B64))) as archive:
    for member in archive.infolist():
        (WORK / member.filename).resolve().relative_to(WORK.resolve())
    archive.extractall(WORK)
print('Current training/inference code applied.')


In [ ]:
# Use Kaggle-hosted weights only after verifying the pinned upstream checkpoint.
# Kaggle's host Python is 3.13; pinned IndicF5 uses an isolated Python 3.11 runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
ENV = Path('/tmp/jeet_tts_env')
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', str(ENV)], check=True)
TRAIN_PYTHON = str(ENV / 'bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', TRAIN_PYTHON, 'torch==2.5.1', 'torchaudio==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', TRAIN_PYTHON, '-r', 'requirements-tts.txt'], check=True)
import hashlib
EXPECTED_WEIGHTS_SHA256 = 'ba7f3671180fb7784e24bd1dafc96e729a38ce02e7f6d3877cdef32525a1865c'
EXPECTED_VOCAB_SHA256 = 'd3a5ff6aac12ea4fb50628a66a97b20c1e83b9e5ca356e5afae17b511cda96df'
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
weights = next((p for p in INPUT.rglob('model.safetensors') if p.is_file() and sha256_file(p) == EXPECTED_WEIGHTS_SHA256), None)
vocab = next((p for p in INPUT.rglob('vocab.txt') if p.is_file() and sha256_file(p) == EXPECTED_VOCAB_SHA256), None)
assert weights is not None and vocab is not None, 'Attached base files do not match the pinned IndicF5 checkpoint/vocabulary; training stopped.'
base = Path('/tmp/jeet_indicf5_verified')
base.mkdir()
(base / 'model.safetensors').symlink_to(weights)
(base / 'vocab.txt').symlink_to(vocab)
os.environ['SAMVYAS_INDICF5_DIR'] = str(base)
# Avoid inheriting a Jupyter-only matplotlib backend into the isolated CLI.
os.environ['MPLBACKEND'] = 'Agg'
(WORK / 'base_verification.json').write_text(json.dumps({'weights_sha256': EXPECTED_WEIGHTS_SHA256, 'vocab_sha256': EXPECTED_VOCAB_SHA256, 'verified': True}, indent=2))
subprocess.run([TRAIN_PYTHON, '-c', "import torch; assert torch.cuda.is_available(), 'GPU required'; print(torch.cuda.get_device_name(0))"], check=True)
print('Pinned base checkpoint and vocabulary verified.', flush=True)


In [ ]:
# Batch one, effective batch four. T4: FP16 + scaler; compatible GPUs: BF16.
subprocess.run([TRAIN_PYTHON, 'scripts/train_indic_f5.py', '--data-root', 'dataset', '--output-dir', 'checkpoints', '--epochs', '40', '--last-blocks', '2', '--learning-rate', '0.00001', '--accumulation', '4', '--patience', '5'], check=True)
print((WORK / 'checkpoints/training_report.json').read_text())


In [ ]:
# Use a training-only reference so held-out recordings remain unseen.
meta = json.loads((WORK / 'dataset/dataset.json').read_text())
rows = [json.loads(line) for line in (WORK / 'dataset/train.jsonl').read_text().splitlines()]
reference = next(row for row in rows if row['clip_id'] == meta['reference_clip_id'])
transcript = WORK / 'reference.txt'
transcript.write_text(reference['text'], encoding='utf-8')
common = [TRAIN_PYTHON, 'scripts/test_indic_f5.py', '--reference', str(WORK / 'dataset' / reference['audio_path']), '--reference-text-file', str(transcript), '--device', 'cuda']
subprocess.run(common + ['--checkpoint', 'checkpoints/best.safetensors', '--output-dir', 'listening/adapted'], check=True)
subprocess.run(common + ['--output-dir', 'listening/base'], check=True)


In [ ]:
from IPython.display import Audio, display
for index in range(1, 4):
    for version in ['base', 'adapted']:
        print(version, index)
        display(Audio(filename=str(WORK / f'listening/{version}/baseline_{index:02d}_24k.wav')))
# Compare pronunciation, speaker likeness, pitch/rhythm, and artifacts.
# Lower flow-matching loss is not proof of better voice similarity.


## Export
Download `checkpoints/best.safetensors` for local inference; keep `last.safetensors` and reports for comparison. These checkpoints contain full online weights; the `ema_model.` naming is for loader compatibility and does not imply EMA averaging. Runs do not resume optimizer state. For a new run use a new output folder. Keep the pinned IndicF5 vocabulary and Jeet reference/transcript when synthesizing.
Local command (after downloading):
```powershell
.\.venv-tts\Scripts\python.exe scripts/test_indic_f5.py --checkpoint PATH_TO_best.safetensors --dataset assets/audio/voice_users/jeet --clip-id e2b80c70c62d48d99f6bfec0e9b8df07 --cpu-fallback --output-dir outputs/tts/jeet_adapted_preview
```
Training and generated audio will stay under `/kaggle/working/jeet_adaptation`. This preparation does not mean training has already run on Kaggle.
